# Project Hive - Kaggle runner

Run the cells top to bottom.

**Before you start:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100)
2. Settings -> **Internet: On** (needed to clone the repo and pull Qwen3-4B)
3. Add your LoRA adapter as a Kaggle **Dataset** input (the `adapter_config.json` +
   `adapter_model.safetensors` pair). Cell 3 finds it automatically.


## 1. Clone the repo

Always syncs hard to `origin/main` -- if this cell already ran earlier in the session,
it discards the local `sed` patch below and re-fetches, so the latest fixes on GitHub
always land here even on a second run in the same kernel.


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed:\n{r.stdout}\n{r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])

# Hard sync to origin/main every time this cell runs -- a plain `git pull` can fail
# silently (or refuse) once the sed patch below has dirtied main.py locally, which
# would leave a re-run of this cell silently stuck on stale code. fetch + reset
# --hard is deterministic regardless of any local edits from a previous run.
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])

# main.py's intra-project imports are live in the repo now (guarded by
# tests/test_main_importable.py); this sed is a no-op safety net kept only for
# older commits where they were still commented out. Harmless and idempotent.
run(['sed', '-i', 's/^# from /from /', REPO_DIR + '/main.py'])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip()
print('synced to:', head)
print(sorted(f for f in os.listdir(REPO_DIR) if f.endswith('.py')))


## 2. Install dependencies

Kaggle already ships `torch`, `numpy` and `pandas`. If pip upgrades something that was
already imported, restart the session (Run -> Restart) and re-run from cell 1.


In [ ]:
!pip -q install -U transformers peft accelerate sentence-transformers faiss-cpu sympy


## 3. Config

The glob auto-detects your adapter under `/kaggle/input`. Hardcode `ADAPTER_PATH` if you
have more than one adapter attached.


In [ ]:
import glob

MODEL_NAME = 'Qwen/Qwen3-4B'
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
GHOST_PERSIST_PATH = '/kaggle/working/hive_memory/ghosts'   # failure memory, persists across runs

ADAPTER_PATH = None      # <-- e.g. '/kaggle/input/adapter-model-v1' to skip autodetect

if ADAPTER_PATH is None:
    hits = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
    if not hits:
        raise FileNotFoundError(
            'No adapter_config.json under /kaggle/input. Add your LoRA adapter as a '
            'dataset input, or set ADAPTER_PATH manually.')
    if len(hits) > 1:
        print('Multiple adapters found, using the first:')
        for h in hits:
            print('  ', h)
    ADAPTER_PATH = os.path.dirname(hits[0])

os.makedirs(os.path.dirname(GHOST_PERSIST_PATH), exist_ok=True)
print('adapter :', ADAPTER_PATH)
print('files   :', sorted(os.listdir(ADAPTER_PATH)))


## 4. Load the model (slow - a few minutes, once per session)

Base model + LoRA adapter + embedder load here and stay in memory. Every colony run
afterwards reuses them, so you pay this cost once.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from peft import PeftModel

import main   # for build_llm_call_fn + the VRAM reporter
from colony_state import ColonyState
from task_graph import TaskGraph
from event_queue import Messenger
from problem_phaser import Problem_Phaser
from judge import Judge
from memory_state import MemoryStore
from synthesizer import Synthesizer
from orchestrator import Orchestrator

print('Loading base model:', MODEL_NAME, '...')
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokeniser.pad_token_id is None:
    tokeniser.pad_token = tokeniser.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto')

# Kaggle ships torchao 0.10.0; the peft installed in cell 2 wants >=0.16.0, and its
# availability check raises rather than returning False. Nothing here uses torchao
# quantization - the LoRA dispatcher only probes for it - so tell peft it's absent.
import peft.tuners.lora.torchao as _lora_torchao
_lora_torchao.is_torchao_available = lambda: False

print('Loading LoRA adapter:', ADAPTER_PATH, '...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

if hasattr(model, 'hf_device_map'):
    devices = set(model.hf_device_map.values())
    print('[DEVICE MAP] layers on:', devices)
    if any(str(d) == 'cpu' for d in devices):
        print('[DEVICE MAP] WARNING: CPU offload detected - generation will be very slow.')

print('Loading embedder:', EMBED_MODEL_NAME, '...')
embedder = SentenceTransformer(EMBED_MODEL_NAME)

llm_call_fn = main.build_llm_call_fn(model, tokeniser)
main._report_vram('after model + adapter load')
print('Ready.')


## 5. Colony factory

`Orchestrator` carries live per-run state (task graph, colony budget, event bus), so each
prompt gets a **fresh** one. Model, embedder and failure memory are shared across runs.


In [ ]:
memory_store = MemoryStore(ghost_persist_path=GHOST_PERSIST_PATH, embed_model=embedder)

def new_colony():
    """Fresh orchestrator wired to the already-loaded model/embedder/memory."""
    return Orchestrator(
        ColonyState(initial_budget=0, goal_embedding=None),
        TaskGraph(),
        Messenger(),
        phaser=Problem_Phaser(model, tokeniser, embed_model=embedder),
        judge=Judge(llm_call_fn=llm_call_fn),
        memory_store=memory_store,
        synthesizer=Synthesizer(llm_call_fn=llm_call_fn),
        model=model,
        tokeniser=tokeniser,
        embed_model=embedder,
    )

print('colony factory ready')


## 6. Your prompt

Edit this cell, then re-run cells 6 and 7 for each new question - no model reload needed.


In [ ]:
PROMPT = """Design a rate limiter for an API gateway handling 50k requests per second
across 12 regions, with per-customer quotas and a hard requirement that no customer
can be throttled by another customer's traffic spike."""

print(PROMPT)


## 7. Run the colony


In [ ]:
import gc

orch = new_colony()
try:
    final_answer = orch.run(PROMPT)
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    main._report_vram('after colony terminate')

print('\n=== FINAL ANSWER ===\n')
print(final_answer)


## 8. (Optional) Save the answer, inspect the failure memory


In [ ]:
with open('/kaggle/working/hive_answer.txt', 'w', encoding='utf-8') as f:
    f.write('PROMPT:\n' + PROMPT + '\n\nANSWER:\n' + str(final_answer) + '\n')
print('saved -> /kaggle/working/hive_answer.txt')

# ghosts = distilled lessons from agents that died, this run and previous ones
print('ghost records on disk:', sorted(glob.glob(GHOST_PERSIST_PATH + '*')))
